In [ ]:
#@title LCMS Method Summary (ZIP Upload)
#@markdown ## LCMS Method Summary (ZIP Upload)
#@markdown **Quick Instructions**
#@markdown 1. On your local machine, zip your method folder(s).
#@markdown    - You can zip each `*.m` folder separately, or
#@markdown    - put multiple `*.m` folders into one ZIP file.
#@markdown 2. Run this cell (hit Runtime -> Run all).
#@markdown 3. Click the **Choose Files** button and select your `.zip` file(s).
#@markdown    - Note: it may take a few seconds for the button to appear.
#@markdown 4. The script runs automatically and writes:
#@markdown    - `/content/sample_data/lc_method_summary.csv`
#@markdown    - `/content/sample_data/lc_method_summary.html`
#@markdown 5. HTML preview + file downloads are triggered automatically.
#@markdown ---
#@markdown ZIPs must contain one or more `*.m` method folders.

# Colab one-cell workflow: includes viewer code inline + ZIP upload run
import sys
import subprocess
import tempfile
import zipfile
from pathlib import Path


def _ensure_package(pkg: str) -> None:
    try:
        __import__(pkg)
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


for _pkg in ("pandas", "plotly"):
    _ensure_package(_pkg)

try:
    from google.colab import files  # type: ignore
except Exception as exc:
    raise RuntimeError(f"This notebook is intended for Google Colab: {exc}")

from __future__ import annotations

import hashlib
import json
import re
import sys
from dataclasses import asdict, dataclass
from datetime import datetime
from pathlib import Path
from typing import Any
import xml.etree.ElementTree as ET

import pandas as pd


REQUIRED_XML_FILES = ("info.xml", "BinPump_1.xml", "ALS_1.xml", "TCC_1.xml")
OUTPUT_CSV = "lc_method_summary.csv"
OUTPUT_HTML = "lc_method_summary.html"
QTOF_REPORT_PATTERN = re.compile(r"<\?xml[^>]*\?>\s*<QTOFTuneReport[\s\S]*?</QTOFTuneReport>")
CFBF_SIGNATURE = b"\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1"
CFBF_FREESECT = 0xFFFFFFFF
CFBF_ENDOFCHAIN = 0xFFFFFFFE
DEFAULT_UNCHECKED_COLUMNS = {
    "description",
    "created_at",
    "modified_at",
    "estimated_run_time_min",
    "low_pressure_limit_bar",
    "high_pressure_limit_bar",
    "max_flow_ramp",
    "initial_percent_a",
    "initial_percent_b",
    "gradient_profile",
    "injection_mode",
    "needle_wash_location",
    "needle_wash_time_s",
    "draw_speed",
    "eject_speed",
    "wait_time_after_draw_s",
    "draw_position_offset",
    "sample_flush_out_factor",
    "valve_to_bypass_after_injection",
    "left_temp_mode",
    "left_not_ready_limit",
    "right_temp_mode",
    "ms_parse_status",
    "stg_size_bytes",
    "stg_md5",
    "ms_report_count",
    "ms_available_polarities",
    # "ms_selected_polarity",
    "ms_model",
    "ms_serial_number",
    "ms_firmware_version",
    # "ms_source_type",
    "ms_suremass",
    # "ms_min_range_mz",
    # "ms_max_range_mz",
    # "ms_acquisition_rate_hz",
    "ms_acquisition_time_ms",
    "ms_method_parse_status",
    "ms_method_progid",
    "ms_method_com_class",
    "ms_method_type_id",
    "ms_method_default_tune_file",
    "ms_method_targeted_list_file",
    "ms_method_profile_token",
    "ms_method_model_token",
    "ms_tune_report_type",
    "ms_tune_report_title",
    "ms_tune_data_path",
    "ms_tune_user",
    "ms_tune_last_user",
    "ms_tune_slicer_mode",
    "ms_tune_state_save_delta",
    "ms_tune_state_gain_channel",
    "ms_tune_state_high_resolution_enabled",
    "ms_tune_cal_a",
    "ms_tune_cal_b",
    "ms_tune_cal_c",
    "ms_tune_cal_d",
    "ms_tune_cal_e",
    "ms_tune_cal_f",
    "ms_tune_cal_poly_terms_flag",
    "ms_tune_cal_trad_weighting",
    "ms_tune_cal_poly_weighting",
    "ms_tune_vac_quad_temp_C",
    "ms_tune_vac_rough_torr",
    "ms_tune_vac_high_torr",
    "ms_tune_vac_octknee_torr",
    "ms_tune_vac_turbo1_pwr_w",
    "ms_tune_vac_turbo1_spd_pct",
    "ms_tune_vac_turbo2_pwr_w",
    "ms_tune_vac_turbo2_spd_pct",
    "ms_tune_cal_point_count",
    "ms_tune_expected_mass_min_mz",
    "ms_tune_expected_mass_max_mz",
    "ms_tune_resolution_mean",
    "ms_tune_resolution_min",
    "ms_tune_resolution_max",
    "ms_set_opt1_fragment_V",
    "ms_set_opt1_skim1_V",
    "ms_set_opt1_oct_peak_V",
    "ms_set_opt1_lens1_V",
    "ms_set_opt1_lens2_V",
    "ms_set_quad_mass_dac_amu",
    "ms_set_quad_quad_dc_V",
    "ms_set_cell_cc_flow_psi",
    "ms_set_cell_hex_rf_V",
    "ms_set_cell_skim2_V",
    "ms_set_opt2_extractor_dc_V",
    "ms_set_opt2_ion_focus_V",
    "ms_set_opt2_slit_left_V",
    "ms_set_opt2_slit_right_V",
    "ms_set_tof_vpulse_V",
    "ms_set_tof_pusher_offset_mV",
    "ms_set_tof_vliner_V",
    "ms_set_acq_num_pts",
    "ms_set_acq_num_trans",
    "ms_set_det_vmcpback_V",
    "ms_set_det_preamp_offset",
    "ms_set_det_dualgain_ratio",
    "ms_act_src_corona_vol_V",
    "ms_act_src_cap_cur",
    "ms_act_src_cham_cur",
}


@dataclass(frozen=True)
class GradientPoint:
    time_min: float
    percent_b: float


@dataclass
class MethodSummary:
    method_dir: str
    description: str | None
    created_at: str | None
    modified_at: str | None
    estimated_run_time_min: float | None
    flow_mL_min: float | None
    stop_time_min: float | None
    post_time_min: float | None
    low_pressure_limit_bar: float | None
    high_pressure_limit_bar: float | None
    max_flow_ramp: float | None
    solvent_a_name: str | None
    solvent_b_name: str | None
    initial_percent_a: float | None
    initial_percent_b: float | None
    gradient_profile: str
    gradient_profile_json: str
    injection_mode: str | None
    injection_volume_uL: float | None
    needle_wash_location: str | None
    needle_wash_time_s: float | None
    draw_speed: float | None
    eject_speed: float | None
    wait_time_after_draw_s: float | None
    draw_position_offset: float | None
    sample_flush_out_factor: float | None
    valve_to_bypass_after_injection: str | None
    column_valve_position: str | None
    left_temp_mode: str | None
    left_temp_C: float | None
    left_not_ready_limit: float | None
    right_temp_mode: str | None
    ms_parse_status: str
    stg_size_bytes: int | None
    stg_md5: str | None
    ms_report_count: int | None
    ms_available_polarities: str | None
    ms_selected_polarity: str | None
    ms_tune_datetime: str | None
    ms_model: str | None
    ms_serial_number: str | None
    ms_firmware_version: str | None
    ms_source_type: str | None
    ms_mass_range: str | None
    ms_instrument_mode: str | None
    ms_suremass: str | None
    ms_min_range_mz: float | None
    ms_max_range_mz: float | None
    ms_acquisition_rate_hz: float | None
    ms_acquisition_time_ms: float | None
    ms_set_src_gas_temp_C: float | None
    ms_set_src_drying_gas_L_min: float | None
    ms_set_src_neb_psi: float | None
    ms_set_src_vcap_V: float | None
    ms_set_src_nozzle_voltage_V: float | None
    ms_set_src_sheath_gas_temp_C: float | None
    ms_set_src_sheath_gas_flow_L_min: float | None
    ms_act_src_gas_temp_C: float | None
    ms_act_src_vapor_temp_C: float | None
    ms_act_src_drying_gas_L_min: float | None
    ms_act_src_neb_psi: float | None
    ms_act_src_vcap_V: float | None
    ms_tune_report_type: str | None
    ms_tune_report_title: str | None
    ms_tune_data_path: str | None
    ms_tune_user: str | None
    ms_tune_last_user: str | None
    ms_tune_slicer_mode: str | None
    ms_tune_state_save_delta: str | None
    ms_tune_state_gain_channel: str | None
    ms_tune_state_high_resolution_enabled: str | None
    ms_tune_cal_a: float | None
    ms_tune_cal_b: float | None
    ms_tune_cal_c: float | None
    ms_tune_cal_d: float | None
    ms_tune_cal_e: float | None
    ms_tune_cal_f: float | None
    ms_tune_cal_poly_terms_flag: str | None
    ms_tune_cal_trad_weighting: float | None
    ms_tune_cal_poly_weighting: float | None
    ms_tune_vac_quad_temp_C: float | None
    ms_tune_vac_rough_torr: float | None
    ms_tune_vac_high_torr: float | None
    ms_tune_vac_octknee_torr: float | None
    ms_tune_vac_turbo1_pwr_w: float | None
    ms_tune_vac_turbo1_spd_pct: float | None
    ms_tune_vac_turbo2_pwr_w: float | None
    ms_tune_vac_turbo2_spd_pct: float | None
    ms_tune_cal_point_count: int | None
    ms_tune_expected_mass_min_mz: float | None
    ms_tune_expected_mass_max_mz: float | None
    ms_tune_resolution_mean: float | None
    ms_tune_resolution_min: float | None
    ms_tune_resolution_max: float | None
    ms_mode_hint: str
    ms_set_opt1_fragment_V: float | None
    ms_set_opt1_skim1_V: float | None
    ms_set_opt1_oct_peak_V: float | None
    ms_set_opt1_lens1_V: float | None
    ms_set_opt1_lens2_V: float | None
    ms_set_quad_mass_dac_amu: float | None
    ms_set_quad_quad_dc_V: float | None
    ms_set_cell_cc_flow_psi: float | None
    ms_set_cell_hex_rf_V: float | None
    ms_set_cell_skim2_V: float | None
    ms_set_opt2_extractor_dc_V: float | None
    ms_set_opt2_ion_focus_V: float | None
    ms_set_opt2_slit_left_V: float | None
    ms_set_opt2_slit_right_V: float | None
    ms_set_tof_vpulse_V: float | None
    ms_set_tof_pusher_offset_mV: float | None
    ms_set_tof_vliner_V: float | None
    ms_set_acq_num_pts: int | None
    ms_set_acq_num_trans: float | None
    ms_set_det_vmcpback_V: float | None
    ms_set_det_preamp_offset: float | None
    ms_set_det_dualgain_ratio: float | None
    ms_act_src_corona_vol_V: float | None
    ms_act_src_cap_cur: float | None
    ms_act_src_cham_cur: float | None
    ms_method_parse_status: str
    ms_method_progid: str | None
    ms_method_com_class: str | None
    ms_method_type_id: str | None
    ms_method_default_tune_file: str | None
    ms_method_targeted_list_file: str | None
    ms_method_profile_token: str | None
    ms_method_model_token: str | None


def warn(message: str) -> None:
    print(f"[WARN] {message}", file=sys.stderr)


def parse_xml(path: Path) -> ET.Element:
    return ET.parse(path).getroot()


def text_at(root: ET.Element, path: str, default: str | None = None) -> str | None:
    node = root.find(".//" + path)
    if node is None:
        return default
    text = (node.text or "").strip()
    return text if text else default


def to_float(value: str | None) -> float | None:
    if value is None:
        return None
    try:
        normalized = value.strip().replace(",", "")
        return float(normalized)
    except ValueError:
        return None


def mean_or_none(values: list[float]) -> float | None:
    if not values:
        return None
    return sum(values) / len(values)


def infer_ms_mode_hint(msmethod_text: str) -> str:
    msms_patterns = [
        r"\bms\s*/\s*ms\b",
        r"\bmsms\b",
        r"\bauto\s*ms\b",
        r"\bautoms\b",
        r"\ball[\s_-]*ions?\b",
        r"\bdia\b",
        r"\bdda\b",
        r"\bmrm\b",
        r"\bsrm\b",
        r"\bprm\b",
        r"\bprecursor\b",
        r"\bproduct[\s_-]*ion\b",
    ]
    scan_patterns = [
        r"\bscanmode\b",
        r"\bfull[\s_-]*scan\b",
        r"\bscan\b",
    ]

    for pat in msms_patterns:
        if re.search(pat, msmethod_text, flags=re.IGNORECASE):
            return "msms_likely_from_method_folder"

    for pat in scan_patterns:
        if re.search(pat, msmethod_text, flags=re.IGNORECASE):
            return "scan_likely_from_method_folder"

    return "unknown_from_method_folder"


def parse_expected_polarity(method_dir_name: str) -> str | None:
    name = method_dir_name.lower()
    if "_pos" in name:
        return "Positive"
    if "_neg" in name:
        return "Negative"
    return None


def extract_ascii_strings(data: bytes, min_len: int = 4, max_len: int = 160) -> list[str]:
    pattern = re.compile(rb"[ -~]{%d,}" % min_len)
    out: list[str] = []
    seen: set[str] = set()
    for m in pattern.finditer(data):
        s = m.group(0).decode("latin1", errors="ignore").strip()
        if not s or len(s) > max_len or s in seen:
            continue
        seen.add(s)
        out.append(s)
    return out


def parse_cfbf_streams(stg_bytes: bytes) -> dict[str, bytes]:
    if len(stg_bytes) < 512 or stg_bytes[:8] != CFBF_SIGNATURE:
        return {}

    header = stg_bytes[:512]
    sector_shift = int.from_bytes(header[0x1E:0x20], "little")
    mini_sector_shift = int.from_bytes(header[0x20:0x22], "little")
    if sector_shift not in (9, 12) or mini_sector_shift not in (6,):
        return {}

    sector_size = 1 << sector_shift
    mini_sector_size = 1 << mini_sector_shift
    if sector_size <= 0 or mini_sector_size <= 0:
        return {}

    num_fat_sectors = int.from_bytes(header[0x2C:0x30], "little")
    first_dir_sector = int.from_bytes(header[0x30:0x34], "little")
    mini_stream_cutoff = int.from_bytes(header[0x38:0x3C], "little")
    first_mini_fat_sector = int.from_bytes(header[0x3C:0x40], "little")
    num_mini_fat_sectors = int.from_bytes(header[0x40:0x44], "little")
    first_difat_sector = int.from_bytes(header[0x44:0x48], "little")
    num_difat_sectors = int.from_bytes(header[0x48:0x4C], "little")

    def sector_offset(sid: int) -> int:
        return (sid + 1) * sector_size

    def read_sector(sid: int) -> bytes:
        off = sector_offset(sid)
        if off < 0 or off + sector_size > len(stg_bytes):
            return b""
        return stg_bytes[off : off + sector_size]

    # Build DIFAT list (FAT sector IDs).
    fat_sector_ids: list[int] = []
    for i in range(109):
        sid = int.from_bytes(header[0x4C + 4 * i : 0x50 + 4 * i], "little")
        if sid != CFBF_FREESECT:
            fat_sector_ids.append(sid)

    next_difat = first_difat_sector
    for _ in range(num_difat_sectors):
        if next_difat in (CFBF_FREESECT, CFBF_ENDOFCHAIN):
            break
        sec = read_sector(next_difat)
        if len(sec) != sector_size:
            break
        entries_per_difat = sector_size // 4 - 1
        for i in range(entries_per_difat):
            sid = int.from_bytes(sec[4 * i : 4 * (i + 1)], "little")
            if sid != CFBF_FREESECT:
                fat_sector_ids.append(sid)
        next_difat = int.from_bytes(sec[-4:], "little")

    # Build FAT table.
    fat: list[int] = []
    target_fat_ids = fat_sector_ids[:num_fat_sectors] if num_fat_sectors > 0 else fat_sector_ids
    for sid in target_fat_ids:
        sec = read_sector(sid)
        if len(sec) != sector_size:
            continue
        for i in range(0, sector_size, 4):
            fat.append(int.from_bytes(sec[i : i + 4], "little"))
    if not fat:
        return {}

    def read_chain(start_sid: int, table: list[int], chunk_size: int, source: bytes | None = None, size: int | None = None) -> bytes:
        out = bytearray()
        sid = start_sid
        visited: set[int] = set()
        while (
            sid not in (CFBF_ENDOFCHAIN, CFBF_FREESECT)
            and 0 <= sid < len(table)
            and sid not in visited
        ):
            visited.add(sid)
            if source is None:
                chunk = read_sector(sid)
            else:
                off = sid * chunk_size
                chunk = source[off : off + chunk_size]
            if not chunk:
                break
            out.extend(chunk)
            sid = table[sid]
        if size is not None:
            return bytes(out[:size])
        return bytes(out)

    # Parse directory entries.
    dir_bytes = read_chain(first_dir_sector, fat, sector_size)
    entries: list[dict[str, Any]] = []
    for i in range(0, len(dir_bytes), 128):
        ent = dir_bytes[i : i + 128]
        if len(ent) < 128:
            break
        name_len = int.from_bytes(ent[64:66], "little")
        if 2 <= name_len <= 64 and name_len % 2 == 0:
            name = ent[: name_len - 2].decode("utf-16le", errors="ignore")
        else:
            name = ""
        entries.append(
            {
                "name": name,
                "obj_type": ent[66],
                "start_sid": int.from_bytes(ent[116:120], "little"),
                "size": int.from_bytes(ent[120:128], "little"),
            }
        )
    if not entries:
        return {}

    root_entry = next((e for e in entries if e["obj_type"] == 5), entries[0])
    root_stream = b""
    if root_entry["start_sid"] not in (CFBF_FREESECT, CFBF_ENDOFCHAIN):
        root_stream = read_chain(root_entry["start_sid"], fat, sector_size, size=root_entry["size"])

    mini_fat: list[int] = []
    if num_mini_fat_sectors > 0 and first_mini_fat_sector not in (CFBF_FREESECT, CFBF_ENDOFCHAIN):
        mini_fat_bytes = read_chain(
            first_mini_fat_sector,
            fat,
            sector_size,
            size=num_mini_fat_sectors * sector_size,
        )
        mini_fat = [
            int.from_bytes(mini_fat_bytes[i : i + 4], "little")
            for i in range(0, len(mini_fat_bytes), 4)
            if i + 4 <= len(mini_fat_bytes)
        ]

    streams: dict[str, bytes] = {}
    for entry in entries:
        if entry["obj_type"] != 2 or not entry["name"]:
            continue
        start_sid = entry["start_sid"]
        size = entry["size"]
        if start_sid in (CFBF_FREESECT, CFBF_ENDOFCHAIN):
            streams[entry["name"]] = b""
            continue
        if size >= mini_stream_cutoff or not mini_fat or not root_stream:
            stream_data = read_chain(start_sid, fat, sector_size, size=size)
        else:
            stream_data = read_chain(start_sid, mini_fat, mini_sector_size, source=root_stream, size=size)
        streams[entry["name"]] = stream_data
    return streams


def parse_msmethod_hints(stg_bytes: bytes) -> dict[str, Any]:
    hints: dict[str, Any] = {
        "ms_method_parse_status": "unparsed",
        "ms_mode_hint": "unknown_from_method_folder",
        "ms_method_progid": None,
        "ms_method_com_class": None,
        "ms_method_type_id": None,
        "ms_method_default_tune_file": None,
        "ms_method_targeted_list_file": None,
        "ms_method_profile_token": None,
        "ms_method_model_token": None,
    }

    streams = parse_cfbf_streams(stg_bytes)
    if not streams:
        hints["ms_method_parse_status"] = "cfbf_parse_failed"
        return hints

    compobj_name = next((name for name in streams if name.lower().endswith("compobj")), None)
    if compobj_name:
        compobj_strings = extract_ascii_strings(streams[compobj_name], min_len=4)
        hints["ms_method_progid"] = next((s for s in compobj_strings if "AgtMSMethod" in s), None)
        hints["ms_method_com_class"] = next((s for s in compobj_strings if "AgtMSSp" in s), None)

    msmethod_data = streams.get("MSMETHOD")
    if not msmethod_data:
        hints["ms_method_parse_status"] = "msmethod_stream_missing"
        return hints

    ms_u16 = msmethod_data.decode("utf-16le", errors="ignore")
    ms_l1 = msmethod_data.decode("latin1", errors="ignore")
    hints["ms_mode_hint"] = infer_ms_mode_hint(f"{ms_u16}\n{ms_l1}")
    tune_match = re.search(r"\b([A-Za-z0-9_.-]+\.tun)\b", ms_u16, flags=re.IGNORECASE)
    csv_match = re.search(r"\b([A-Za-z0-9_.-]+\.csv)\b", ms_u16, flags=re.IGNORECASE)
    model_match = re.search(r"\b(G\d{4}[A-Z]?)\b", ms_u16)
    profile_match = re.search(r"None", ms_u16, flags=re.IGNORECASE)

    hints["ms_method_default_tune_file"] = tune_match.group(1) if tune_match else None
    hints["ms_method_targeted_list_file"] = csv_match.group(1) if csv_match else None
    hints["ms_method_model_token"] = model_match.group(1) if model_match else None
    hints["ms_method_profile_token"] = "None" if profile_match else None

    guid_pattern = re.compile(r"\{[0-9A-Fa-f-]{36}\}")
    for stream_data in streams.values():
        ascii_text = stream_data.decode("latin1", errors="ignore")
        guid_match = guid_pattern.search(ascii_text)
        if guid_match:
            hints["ms_method_type_id"] = guid_match.group(0)
            break
        utf16_text = stream_data.decode("utf-16le", errors="ignore")
        guid_match = guid_pattern.search(utf16_text)
        if guid_match:
            hints["ms_method_type_id"] = guid_match.group(0)
            break

    if any(
        hints[key]
        for key in (
            "ms_method_progid",
            "ms_method_com_class",
            "ms_method_type_id",
            "ms_method_default_tune_file",
            "ms_method_targeted_list_file",
            "ms_method_profile_token",
            "ms_method_model_token",
        )
    ):
        hints["ms_method_parse_status"] = "msmethod_hints_parsed"
    else:
        hints["ms_method_parse_status"] = "msmethod_hints_empty"

    return hints


def empty_ms_payload(status: str, size_bytes: int | None, md5_hex: str | None) -> dict[str, Any]:
    return {
        "ms_parse_status": status,
        "stg_size_bytes": size_bytes,
        "stg_md5": md5_hex,
        "ms_report_count": None,
        "ms_available_polarities": None,
        "ms_selected_polarity": None,
        "ms_tune_datetime": None,
        "ms_model": None,
        "ms_serial_number": None,
        "ms_firmware_version": None,
        "ms_source_type": None,
        "ms_mass_range": None,
        "ms_instrument_mode": None,
        "ms_suremass": None,
        "ms_min_range_mz": None,
        "ms_max_range_mz": None,
        "ms_acquisition_rate_hz": None,
        "ms_acquisition_time_ms": None,
        "ms_set_src_gas_temp_C": None,
        "ms_set_src_drying_gas_L_min": None,
        "ms_set_src_neb_psi": None,
        "ms_set_src_vcap_V": None,
        "ms_set_src_nozzle_voltage_V": None,
        "ms_set_src_sheath_gas_temp_C": None,
        "ms_set_src_sheath_gas_flow_L_min": None,
        "ms_act_src_gas_temp_C": None,
        "ms_act_src_vapor_temp_C": None,
        "ms_act_src_drying_gas_L_min": None,
        "ms_act_src_neb_psi": None,
        "ms_act_src_vcap_V": None,
        "ms_tune_report_type": None,
        "ms_tune_report_title": None,
        "ms_tune_data_path": None,
        "ms_tune_user": None,
        "ms_tune_last_user": None,
        "ms_tune_slicer_mode": None,
        "ms_tune_state_save_delta": None,
        "ms_tune_state_gain_channel": None,
        "ms_tune_state_high_resolution_enabled": None,
        "ms_tune_cal_a": None,
        "ms_tune_cal_b": None,
        "ms_tune_cal_c": None,
        "ms_tune_cal_d": None,
        "ms_tune_cal_e": None,
        "ms_tune_cal_f": None,
        "ms_tune_cal_poly_terms_flag": None,
        "ms_tune_cal_trad_weighting": None,
        "ms_tune_cal_poly_weighting": None,
        "ms_tune_vac_quad_temp_C": None,
        "ms_tune_vac_rough_torr": None,
        "ms_tune_vac_high_torr": None,
        "ms_tune_vac_octknee_torr": None,
        "ms_tune_vac_turbo1_pwr_w": None,
        "ms_tune_vac_turbo1_spd_pct": None,
        "ms_tune_vac_turbo2_pwr_w": None,
        "ms_tune_vac_turbo2_spd_pct": None,
        "ms_tune_cal_point_count": None,
        "ms_tune_expected_mass_min_mz": None,
        "ms_tune_expected_mass_max_mz": None,
        "ms_tune_resolution_mean": None,
        "ms_tune_resolution_min": None,
        "ms_tune_resolution_max": None,
        "ms_mode_hint": "unknown_from_method_folder",
        "ms_set_opt1_fragment_V": None,
        "ms_set_opt1_skim1_V": None,
        "ms_set_opt1_oct_peak_V": None,
        "ms_set_opt1_lens1_V": None,
        "ms_set_opt1_lens2_V": None,
        "ms_set_quad_mass_dac_amu": None,
        "ms_set_quad_quad_dc_V": None,
        "ms_set_cell_cc_flow_psi": None,
        "ms_set_cell_hex_rf_V": None,
        "ms_set_cell_skim2_V": None,
        "ms_set_opt2_extractor_dc_V": None,
        "ms_set_opt2_ion_focus_V": None,
        "ms_set_opt2_slit_left_V": None,
        "ms_set_opt2_slit_right_V": None,
        "ms_set_tof_vpulse_V": None,
        "ms_set_tof_pusher_offset_mV": None,
        "ms_set_tof_vliner_V": None,
        "ms_set_acq_num_pts": None,
        "ms_set_acq_num_trans": None,
        "ms_set_det_vmcpback_V": None,
        "ms_set_det_preamp_offset": None,
        "ms_set_det_dualgain_ratio": None,
        "ms_act_src_corona_vol_V": None,
        "ms_act_src_cap_cur": None,
        "ms_act_src_cham_cur": None,
        "ms_method_parse_status": "unparsed",
        "ms_method_progid": None,
        "ms_method_com_class": None,
        "ms_method_type_id": None,
        "ms_method_default_tune_file": None,
        "ms_method_targeted_list_file": None,
        "ms_method_profile_token": None,
        "ms_method_model_token": None,
    }


def parse_ms_stg(path: Path, expected_polarity: str | None) -> dict[str, Any]:
    if not path.exists():
        return empty_ms_payload("stg_missing", None, None)

    stg_bytes = path.read_bytes()
    stg_md5 = hashlib.md5(stg_bytes).hexdigest()
    stg_size = path.stat().st_size
    method_hints = parse_msmethod_hints(stg_bytes)

    try:
        decoded = stg_bytes.decode("utf-16le", errors="ignore")
    except Exception:
        payload = empty_ms_payload("stg_decode_failed", stg_size, stg_md5)
        payload.update(method_hints)
        return payload

    report_texts = QTOF_REPORT_PATTERN.findall(decoded)
    if not report_texts:
        payload = empty_ms_payload("qtof_xml_not_found", stg_size, stg_md5)
        payload.update(method_hints)
        return payload

    report_roots: list[ET.Element] = []
    seen_report_keys: set[tuple[str, str, str]] = set()
    for report_text in report_texts:
        try:
            root = ET.fromstring(report_text)
        except ET.ParseError:
            continue

        key = (
            text_at(root, "polarity") or "",
            text_at(root, "TuneDateTime") or "",
            text_at(root, "TuneReportType") or "",
        )
        if key in seen_report_keys:
            continue
        seen_report_keys.add(key)
        report_roots.append(root)

    if not report_roots:
        payload = empty_ms_payload("qtof_xml_parse_failed", stg_size, stg_md5)
        payload.update(method_hints)
        return payload

    def report_sort_key(root: ET.Element) -> str:
        return text_at(root, "TuneDateTime") or ""

    selected_root: ET.Element | None = None
    fallback_used = False
    if expected_polarity:
        matching = [
            root
            for root in report_roots
            if (text_at(root, "polarity") or "").strip().lower() == expected_polarity.lower()
        ]
        if matching:
            selected_root = max(matching, key=report_sort_key)
        else:
            fallback_used = True

    if selected_root is None:
        selected_root = max(report_roots, key=report_sort_key)

    expected_masses: list[float] = []
    resolutions: list[float] = []
    for mass_node in selected_root.findall(".//TTISpectrum/MassList"):
        expected_mass = to_float(text_at(mass_node, "ExpectedMass"))
        if expected_mass is not None:
            expected_masses.append(expected_mass)
        resolution = to_float(text_at(mass_node, "Resolution"))
        if resolution is not None:
            resolutions.append(resolution)

    polarities = sorted({(text_at(root, "polarity") or "").strip() for root in report_roots if text_at(root, "polarity")})
    payload = {
        "ms_parse_status": "parsed_qtof_xml_fallback_polarity" if fallback_used else "parsed_qtof_xml",
        "stg_size_bytes": stg_size,
        "stg_md5": stg_md5,
        "ms_report_count": len(report_roots),
        "ms_available_polarities": ";".join(polarities) if polarities else None,
        "ms_selected_polarity": text_at(selected_root, "polarity"),
        "ms_tune_datetime": text_at(selected_root, "TuneDateTime"),
        "ms_model": text_at(selected_root, "msModel"),
        "ms_serial_number": text_at(selected_root, "SerialNumber"),
        "ms_firmware_version": text_at(selected_root, "FirmwareVersion"),
        "ms_source_type": text_at(selected_root, "sourceType"),
        "ms_mass_range": text_at(selected_root, "MassRange"),
        "ms_instrument_mode": text_at(selected_root, "InstrumentMode"),
        "ms_suremass": text_at(selected_root, "SureMass"),
        "ms_min_range_mz": to_float(text_at(selected_root, "TOFMassRange/MinRangeTune")),
        "ms_max_range_mz": to_float(text_at(selected_root, "TOFMassRange/MaxRangeTune")),
        "ms_acquisition_rate_hz": to_float(text_at(selected_root, "TOFAcquisitionRate/AcquisitionRate")),
        "ms_acquisition_time_ms": to_float(text_at(selected_root, "TOFAcquisitionRate/AcquisitionTime")),
        "ms_set_src_gas_temp_C": to_float(text_at(selected_root, "Setpoints/Source/GasTempTune")),
        "ms_set_src_drying_gas_L_min": to_float(text_at(selected_root, "Setpoints/Source/DryingGasTune")),
        "ms_set_src_neb_psi": to_float(text_at(selected_root, "Setpoints/Source/NebPresTune")),
        "ms_set_src_vcap_V": to_float(text_at(selected_root, "Setpoints/Source/VCap")),
        "ms_set_src_nozzle_voltage_V": to_float(text_at(selected_root, "Setpoints/Source/NozzleVoltage")),
        "ms_set_src_sheath_gas_temp_C": to_float(text_at(selected_root, "Setpoints/Source/SheathGasTempTune")),
        "ms_set_src_sheath_gas_flow_L_min": to_float(text_at(selected_root, "Setpoints/Source/SheathGasFlowTune")),
        "ms_act_src_gas_temp_C": to_float(text_at(selected_root, "Actuals/Source/GasTemp")),
        "ms_act_src_vapor_temp_C": to_float(text_at(selected_root, "Actuals/Source/Vapor")),
        "ms_act_src_drying_gas_L_min": to_float(text_at(selected_root, "Actuals/Source/DryingGas")),
        "ms_act_src_neb_psi": to_float(text_at(selected_root, "Actuals/Source/NebPres")),
        "ms_act_src_vcap_V": to_float(text_at(selected_root, "Actuals/Source/VCapES")),
        "ms_set_opt1_fragment_V": to_float(text_at(selected_root, "Setpoints/Optics1/FragmentTune")),
        "ms_set_opt1_skim1_V": to_float(text_at(selected_root, "Setpoints/Optics1/Skim1Tune")),
        "ms_set_opt1_oct_peak_V": to_float(text_at(selected_root, "Setpoints/Optics1/OctPeakTune")),
        "ms_set_opt1_lens1_V": to_float(text_at(selected_root, "Setpoints/Optics1/Lens1")),
        "ms_set_opt1_lens2_V": to_float(text_at(selected_root, "Setpoints/Optics1/Lens2")),
        "ms_set_quad_mass_dac_amu": to_float(text_at(selected_root, "Setpoints/Quad/MassDAC")),
        "ms_set_quad_quad_dc_V": to_float(text_at(selected_root, "Setpoints/Quad/QuadDC")),
        "ms_set_cell_cc_flow_psi": to_float(text_at(selected_root, "Setpoints/Cell/CCFlow")),
        "ms_set_cell_hex_rf_V": to_float(text_at(selected_root, "Setpoints/Cell/HexRF")),
        "ms_set_cell_skim2_V": to_float(text_at(selected_root, "Setpoints/Cell/Skim2")),
        "ms_set_opt2_extractor_dc_V": to_float(text_at(selected_root, "Setpoints/Optics2/ExtractorDC")),
        "ms_set_opt2_ion_focus_V": to_float(text_at(selected_root, "Setpoints/Optics2/IonFocus")),
        "ms_set_opt2_slit_left_V": to_float(text_at(selected_root, "Setpoints/Optics2/SlitLeft")),
        "ms_set_opt2_slit_right_V": to_float(text_at(selected_root, "Setpoints/Optics2/SlitRight")),
        "ms_set_tof_vpulse_V": to_float(text_at(selected_root, "Setpoints/TOF/VPulse")),
        "ms_set_tof_pusher_offset_mV": to_float(text_at(selected_root, "Setpoints/TOF/PusherOffset")),
        "ms_set_tof_vliner_V": to_float(text_at(selected_root, "Setpoints/TOF/VLiner")),
        "ms_set_acq_num_pts": int(to_float(text_at(selected_root, "Setpoints/TOFAcquisitionRate/numPts")) or 0) or None,
        "ms_set_acq_num_trans": to_float(text_at(selected_root, "Setpoints/TOFAcquisitionRate/numTransTune")),
        "ms_set_det_vmcpback_V": to_float(text_at(selected_root, "Setpoints/Detector/Vmcpback")),
        "ms_set_det_preamp_offset": to_float(text_at(selected_root, "Setpoints/Detector/PreAmpOffset")),
        "ms_set_det_dualgain_ratio": to_float(text_at(selected_root, "Setpoints/Detector/DualGainLowHighRatio")),
        "ms_act_src_corona_vol_V": to_float(text_at(selected_root, "Actuals/Source/CoronaVol")),
        "ms_act_src_cap_cur": to_float(text_at(selected_root, "Actuals/Source/CapCur")),
        "ms_act_src_cham_cur": to_float(text_at(selected_root, "Actuals/Source/ChamCur")),
        "ms_tune_report_type": text_at(selected_root, "TuneReportType"),
        "ms_tune_report_title": text_at(selected_root, "TuneReportTitle"),
        "ms_tune_data_path": text_at(selected_root, "TuneDataPathName"),
        "ms_tune_user": text_at(selected_root, "User"),
        "ms_tune_last_user": text_at(selected_root, "lastUser"),
        "ms_tune_slicer_mode": text_at(selected_root, "SlicerMode"),
        "ms_tune_state_save_delta": text_at(selected_root, "InstrumentState/SaveDelta"),
        "ms_tune_state_gain_channel": text_at(selected_root, "InstrumentState/GainChannel"),
        "ms_tune_state_high_resolution_enabled": text_at(selected_root, "InstrumentState/HighResolutionEnabled"),
        "ms_tune_cal_a": to_float(text_at(selected_root, "Calibration/A")),
        "ms_tune_cal_b": to_float(text_at(selected_root, "Calibration/B")),
        "ms_tune_cal_c": to_float(text_at(selected_root, "Calibration/C")),
        "ms_tune_cal_d": to_float(text_at(selected_root, "Calibration/D")),
        "ms_tune_cal_e": to_float(text_at(selected_root, "Calibration/E")),
        "ms_tune_cal_f": to_float(text_at(selected_root, "Calibration/F")),
        "ms_tune_cal_poly_terms_flag": text_at(selected_root, "Calibration/PolyTermsFlag"),
        "ms_tune_cal_trad_weighting": to_float(text_at(selected_root, "Calibration/TradWeighting")),
        "ms_tune_cal_poly_weighting": to_float(text_at(selected_root, "Calibration/PolyWeighting")),
        "ms_tune_vac_quad_temp_C": to_float(text_at(selected_root, "Actuals/QTOF_VacuumTemp/QuadTemp")),
        "ms_tune_vac_rough_torr": to_float(text_at(selected_root, "Actuals/QTOF_VacuumTemp/RoughVac")),
        "ms_tune_vac_high_torr": to_float(text_at(selected_root, "Actuals/QTOF_VacuumTemp/HighVac")),
        "ms_tune_vac_octknee_torr": to_float(text_at(selected_root, "Actuals/QTOF_VacuumTemp/OctKnee")),
        "ms_tune_vac_turbo1_pwr_w": to_float(text_at(selected_root, "Actuals/QTOF_VacuumTemp/Turbo1Pwr")),
        "ms_tune_vac_turbo1_spd_pct": to_float(text_at(selected_root, "Actuals/QTOF_VacuumTemp/Turbo1Spd")),
        "ms_tune_vac_turbo2_pwr_w": to_float(text_at(selected_root, "Actuals/QTOF_VacuumTemp/Turbo2Pwr")),
        "ms_tune_vac_turbo2_spd_pct": to_float(text_at(selected_root, "Actuals/QTOF_VacuumTemp/Turbo2Spd")),
        "ms_tune_cal_point_count": len(expected_masses),
        "ms_tune_expected_mass_min_mz": min(expected_masses) if expected_masses else None,
        "ms_tune_expected_mass_max_mz": max(expected_masses) if expected_masses else None,
        "ms_tune_resolution_mean": mean_or_none(resolutions),
        "ms_tune_resolution_min": min(resolutions) if resolutions else None,
        "ms_tune_resolution_max": max(resolutions) if resolutions else None,
    }
    payload.update(method_hints)
    return payload


def discover_method_dirs(root_dir: Path) -> list[Path]:
    return sorted(path for path in root_dir.glob("*.m") if path.is_dir())


def extract_gradient_points(pump_root: ET.Element) -> list[GradientPoint]:
    initial_percent_b: float | None = None
    entries: list[GradientPoint] = []

    for solvent_element in pump_root.findall(".//SolventComposition/SolventElement"):
        channel = text_at(solvent_element, "Channel")
        percentage = to_float(text_at(solvent_element, "Percentage"))
        if channel == "Channel_B":
            initial_percent_b = percentage
            break

    if initial_percent_b is None:
        initial_percent_b = 0.0

    entries.append(GradientPoint(time_min=0.0, percent_b=initial_percent_b))

    for timetable_entry in pump_root.findall(".//Timetable/TimetableEntry"):
        time_min = to_float(text_at(timetable_entry, "Time"))
        percent_b = to_float(text_at(timetable_entry, "PercentB"))
        if time_min is None or percent_b is None:
            continue
        entries.append(GradientPoint(time_min=time_min, percent_b=percent_b))

    entries.sort(key=lambda x: x.time_min)
    return entries


def build_gradient_profile(points: list[GradientPoint]) -> tuple[str, str]:
    compact = " | ".join(f"{p.time_min:g}:{p.percent_b:g}" for p in points)
    payload = [{"time_min": p.time_min, "percent_b": p.percent_b} for p in points]
    return compact, json.dumps(payload, ensure_ascii=True)


def parse_method_dir(method_dir: Path) -> MethodSummary | None:
    missing_files = [name for name in REQUIRED_XML_FILES if not (method_dir / name).exists()]
    if missing_files:
        warn(f"Skipping {method_dir.name}: missing required files {missing_files}")
        return None

    try:
        info_root = parse_xml(method_dir / "info.xml")
        pump_root = parse_xml(method_dir / "BinPump_1.xml")
        als_root = parse_xml(method_dir / "ALS_1.xml")
        tcc_root = parse_xml(method_dir / "TCC_1.xml")
    except ET.ParseError as exc:
        warn(f"Skipping {method_dir.name}: XML parse error ({exc})")
        return None
    except OSError as exc:
        warn(f"Skipping {method_dir.name}: file read error ({exc})")
        return None

    solvent_a_name = None
    solvent_b_name = None
    initial_percent_a = None
    initial_percent_b = None

    for solvent_element in pump_root.findall(".//SolventComposition/SolventElement"):
        channel = text_at(solvent_element, "Channel")
        name = text_at(solvent_element, "Channel1UserName")
        percentage = to_float(text_at(solvent_element, "Percentage"))
        if channel == "Channel_A":
            solvent_a_name = name
            initial_percent_a = percentage
        elif channel == "Channel_B":
            solvent_b_name = name
            initial_percent_b = percentage

    gradient_points = extract_gradient_points(pump_root)
    gradient_profile, gradient_profile_json = build_gradient_profile(gradient_points)
    expected_polarity = parse_expected_polarity(method_dir.name)
    ms_info = parse_ms_stg(method_dir / "191_1.stg", expected_polarity)

    return MethodSummary(
        method_dir=method_dir.name,
        description=text_at(info_root, "Description"),
        created_at=text_at(info_root, "CreationDate"),
        modified_at=text_at(info_root, "LastModificationDate"),
        estimated_run_time_min=to_float(text_at(info_root, "EstimatedRunTime")),
        flow_mL_min=to_float(text_at(pump_root, "Flow")),
        stop_time_min=to_float(text_at(pump_root, "StopTime/StopTimeValue")),
        post_time_min=to_float(text_at(pump_root, "PostTime/PostTimeValue")),
        low_pressure_limit_bar=to_float(text_at(pump_root, "LowPressureLimit")),
        high_pressure_limit_bar=to_float(text_at(pump_root, "HighPressureLimit")),
        max_flow_ramp=to_float(text_at(pump_root, "MaximumFlowRamp")),
        solvent_a_name=solvent_a_name,
        solvent_b_name=solvent_b_name,
        initial_percent_a=initial_percent_a,
        initial_percent_b=initial_percent_b,
        gradient_profile=gradient_profile,
        gradient_profile_json=gradient_profile_json,
        injection_mode=text_at(als_root, "Injection/InjectionMode"),
        injection_volume_uL=to_float(text_at(als_root, "Injection/InjectionVolume")),
        needle_wash_location=text_at(als_root, "Injection/NeedleWash/NeedleWashLocation"),
        needle_wash_time_s=to_float(text_at(als_root, "Injection/NeedleWash/WashTime")),
        draw_speed=to_float(text_at(als_root, "Auxiliary/DrawSpeed")),
        eject_speed=to_float(text_at(als_root, "Auxiliary/EjectSpeed")),
        wait_time_after_draw_s=to_float(text_at(als_root, "Auxiliary/WaitTimeAfterDraw")),
        draw_position_offset=to_float(text_at(als_root, "Auxiliary/DrawPositionOffset")),
        sample_flush_out_factor=to_float(text_at(als_root, "HighThroughput/SampleFlushOutFactor")),
        valve_to_bypass_after_injection=text_at(als_root, "HighThroughput/ValveToBypassAfterInjection"),
        column_valve_position=text_at(tcc_root, "ColumnValvePosition"),
        left_temp_mode=text_at(tcc_root, "LeftTemperatureControl/TemperatureControlMode"),
        left_temp_C=to_float(text_at(tcc_root, "LeftTemperatureControl/Temperature")),
        left_not_ready_limit=to_float(text_at(tcc_root, "LeftTemperatureControl/NotReadyLimit/NotReadyLimitValue")),
        right_temp_mode=text_at(tcc_root, "RightTemperatureControl/TemperatureControlMode"),
        ms_parse_status=ms_info["ms_parse_status"],
        stg_size_bytes=ms_info["stg_size_bytes"],
        stg_md5=ms_info["stg_md5"],
        ms_report_count=ms_info["ms_report_count"],
        ms_available_polarities=ms_info["ms_available_polarities"],
        ms_selected_polarity=ms_info["ms_selected_polarity"],
        ms_tune_datetime=ms_info["ms_tune_datetime"],
        ms_model=ms_info["ms_model"],
        ms_serial_number=ms_info["ms_serial_number"],
        ms_firmware_version=ms_info["ms_firmware_version"],
        ms_source_type=ms_info["ms_source_type"],
        ms_mass_range=ms_info["ms_mass_range"],
        ms_instrument_mode=ms_info["ms_instrument_mode"],
        ms_suremass=ms_info["ms_suremass"],
        ms_min_range_mz=ms_info["ms_min_range_mz"],
        ms_max_range_mz=ms_info["ms_max_range_mz"],
        ms_acquisition_rate_hz=ms_info["ms_acquisition_rate_hz"],
        ms_acquisition_time_ms=ms_info["ms_acquisition_time_ms"],
        ms_set_src_gas_temp_C=ms_info["ms_set_src_gas_temp_C"],
        ms_set_src_drying_gas_L_min=ms_info["ms_set_src_drying_gas_L_min"],
        ms_set_src_neb_psi=ms_info["ms_set_src_neb_psi"],
        ms_set_src_vcap_V=ms_info["ms_set_src_vcap_V"],
        ms_set_src_nozzle_voltage_V=ms_info["ms_set_src_nozzle_voltage_V"],
        ms_set_src_sheath_gas_temp_C=ms_info["ms_set_src_sheath_gas_temp_C"],
        ms_set_src_sheath_gas_flow_L_min=ms_info["ms_set_src_sheath_gas_flow_L_min"],
        ms_act_src_gas_temp_C=ms_info["ms_act_src_gas_temp_C"],
        ms_act_src_vapor_temp_C=ms_info["ms_act_src_vapor_temp_C"],
        ms_act_src_drying_gas_L_min=ms_info["ms_act_src_drying_gas_L_min"],
        ms_act_src_neb_psi=ms_info["ms_act_src_neb_psi"],
        ms_act_src_vcap_V=ms_info["ms_act_src_vcap_V"],
        ms_set_opt1_fragment_V=ms_info["ms_set_opt1_fragment_V"],
        ms_set_opt1_skim1_V=ms_info["ms_set_opt1_skim1_V"],
        ms_set_opt1_oct_peak_V=ms_info["ms_set_opt1_oct_peak_V"],
        ms_set_opt1_lens1_V=ms_info["ms_set_opt1_lens1_V"],
        ms_set_opt1_lens2_V=ms_info["ms_set_opt1_lens2_V"],
        ms_set_quad_mass_dac_amu=ms_info["ms_set_quad_mass_dac_amu"],
        ms_set_quad_quad_dc_V=ms_info["ms_set_quad_quad_dc_V"],
        ms_set_cell_cc_flow_psi=ms_info["ms_set_cell_cc_flow_psi"],
        ms_set_cell_hex_rf_V=ms_info["ms_set_cell_hex_rf_V"],
        ms_set_cell_skim2_V=ms_info["ms_set_cell_skim2_V"],
        ms_set_opt2_extractor_dc_V=ms_info["ms_set_opt2_extractor_dc_V"],
        ms_set_opt2_ion_focus_V=ms_info["ms_set_opt2_ion_focus_V"],
        ms_set_opt2_slit_left_V=ms_info["ms_set_opt2_slit_left_V"],
        ms_set_opt2_slit_right_V=ms_info["ms_set_opt2_slit_right_V"],
        ms_set_tof_vpulse_V=ms_info["ms_set_tof_vpulse_V"],
        ms_set_tof_pusher_offset_mV=ms_info["ms_set_tof_pusher_offset_mV"],
        ms_set_tof_vliner_V=ms_info["ms_set_tof_vliner_V"],
        ms_set_acq_num_pts=ms_info["ms_set_acq_num_pts"],
        ms_set_acq_num_trans=ms_info["ms_set_acq_num_trans"],
        ms_set_det_vmcpback_V=ms_info["ms_set_det_vmcpback_V"],
        ms_set_det_preamp_offset=ms_info["ms_set_det_preamp_offset"],
        ms_set_det_dualgain_ratio=ms_info["ms_set_det_dualgain_ratio"],
        ms_act_src_corona_vol_V=ms_info["ms_act_src_corona_vol_V"],
        ms_act_src_cap_cur=ms_info["ms_act_src_cap_cur"],
        ms_act_src_cham_cur=ms_info["ms_act_src_cham_cur"],
        ms_tune_report_type=ms_info["ms_tune_report_type"],
        ms_tune_report_title=ms_info["ms_tune_report_title"],
        ms_tune_data_path=ms_info["ms_tune_data_path"],
        ms_tune_user=ms_info["ms_tune_user"],
        ms_tune_last_user=ms_info["ms_tune_last_user"],
        ms_tune_slicer_mode=ms_info["ms_tune_slicer_mode"],
        ms_tune_state_save_delta=ms_info["ms_tune_state_save_delta"],
        ms_tune_state_gain_channel=ms_info["ms_tune_state_gain_channel"],
        ms_tune_state_high_resolution_enabled=ms_info["ms_tune_state_high_resolution_enabled"],
        ms_tune_cal_a=ms_info["ms_tune_cal_a"],
        ms_tune_cal_b=ms_info["ms_tune_cal_b"],
        ms_tune_cal_c=ms_info["ms_tune_cal_c"],
        ms_tune_cal_d=ms_info["ms_tune_cal_d"],
        ms_tune_cal_e=ms_info["ms_tune_cal_e"],
        ms_tune_cal_f=ms_info["ms_tune_cal_f"],
        ms_tune_cal_poly_terms_flag=ms_info["ms_tune_cal_poly_terms_flag"],
        ms_tune_cal_trad_weighting=ms_info["ms_tune_cal_trad_weighting"],
        ms_tune_cal_poly_weighting=ms_info["ms_tune_cal_poly_weighting"],
        ms_tune_vac_quad_temp_C=ms_info["ms_tune_vac_quad_temp_C"],
        ms_tune_vac_rough_torr=ms_info["ms_tune_vac_rough_torr"],
        ms_tune_vac_high_torr=ms_info["ms_tune_vac_high_torr"],
        ms_tune_vac_octknee_torr=ms_info["ms_tune_vac_octknee_torr"],
        ms_tune_vac_turbo1_pwr_w=ms_info["ms_tune_vac_turbo1_pwr_w"],
        ms_tune_vac_turbo1_spd_pct=ms_info["ms_tune_vac_turbo1_spd_pct"],
        ms_tune_vac_turbo2_pwr_w=ms_info["ms_tune_vac_turbo2_pwr_w"],
        ms_tune_vac_turbo2_spd_pct=ms_info["ms_tune_vac_turbo2_spd_pct"],
        ms_tune_cal_point_count=ms_info["ms_tune_cal_point_count"],
        ms_tune_expected_mass_min_mz=ms_info["ms_tune_expected_mass_min_mz"],
        ms_tune_expected_mass_max_mz=ms_info["ms_tune_expected_mass_max_mz"],
        ms_tune_resolution_mean=ms_info["ms_tune_resolution_mean"],
        ms_tune_resolution_min=ms_info["ms_tune_resolution_min"],
        ms_tune_resolution_max=ms_info["ms_tune_resolution_max"],
        ms_mode_hint=ms_info["ms_mode_hint"],
        ms_method_parse_status=ms_info["ms_method_parse_status"],
        ms_method_progid=ms_info["ms_method_progid"],
        ms_method_com_class=ms_info["ms_method_com_class"],
        ms_method_type_id=ms_info["ms_method_type_id"],
        ms_method_default_tune_file=ms_info["ms_method_default_tune_file"],
        ms_method_targeted_list_file=ms_info["ms_method_targeted_list_file"],
        ms_method_profile_token=ms_info["ms_method_profile_token"],
        ms_method_model_token=ms_info["ms_method_model_token"],
    )


def build_gradient_traces(df: pd.DataFrame) -> list[dict[str, Any]]:
    traces: list[dict[str, Any]] = []
    for _, row in df.iterrows():
        try:
            profile = json.loads(row["gradient_profile_json"])
        except (TypeError, json.JSONDecodeError):
            continue
        if not profile:
            continue
        x = [point["time_min"] for point in profile]
        y = [point["percent_b"] for point in profile]
        traces.append(
            {
                "x": x,
                "y": y,
                "mode": "lines+markers",
                "line": {"shape": "linear"},
                "name": row["method_dir"],
                "type": "scatter",
            }
        )
    return traces


def to_csv(df: pd.DataFrame, output_path: Path) -> None:
    df.to_csv(output_path, index=False)


def to_html(df: pd.DataFrame, traces: list[dict[str, Any]], output_path: Path) -> None:
    table_df = df.drop(columns=["gradient_profile_json"])
    ms_column_set = {"ms_parse_status", "stg_size_bytes", "stg_md5"}
    ms_column_set.update(col for col in table_df.columns if col.startswith("ms_"))
    ms_method_column_set = {
        col
        for col in table_df.columns
        if col.startswith("ms_method_") or col.startswith("ms_set_") or col.startswith("ms_act_") or col.startswith("ms_mode_")
    }

    lc_columns = [col for col in table_df.columns if col not in ms_column_set]
    ms_tune_columns = [
        "method_dir"
    ] + [
        col
        for col in table_df.columns
        if col != "method_dir" and col in ms_column_set and col not in ms_method_column_set
    ]
    ms_method_columns = ["method_dir"] + [col for col in table_df.columns if col != "method_dir" and col in ms_method_column_set]

    lc_table_df = table_df[lc_columns]
    ms_tune_table_df = table_df[ms_tune_columns]
    ms_method_table_df = table_df[ms_method_columns]

    lc_table_html = lc_table_df.to_html(
        index=False,
        escape=True,
        table_id="lc_method_table",
        classes=["method-table"],
    )
    ms_tune_table_html = ms_tune_table_df.to_html(
        index=False,
        escape=True,
        table_id="ms_tune_table",
        classes=["method-table"],
    )
    ms_method_table_html = ms_method_table_df.to_html(
        index=False,
        escape=True,
        table_id="ms_method_table",
        classes=["method-table"],
    )
    generated_at = datetime.now().isoformat(timespec="seconds")
    traces_json = json.dumps(traces, ensure_ascii=True)
    unchecked_columns_json = json.dumps(sorted(DEFAULT_UNCHECKED_COLUMNS), ensure_ascii=True)

    html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1.0" />
  <title>LCMS Method Summary</title>
  <script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
  <style>
    body {{
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
      margin: 24px;
      color: #222;
      background: #fafafa;
    }}
    h1 {{
      margin-bottom: 4px;
    }}
    .meta {{
      color: #555;
      margin-bottom: 16px;
    }}
    .panel {{
      background: #fff;
      border: 1px solid #ddd;
      border-radius: 8px;
      padding: 16px;
      margin-bottom: 20px;
      box-shadow: 0 1px 3px rgba(0, 0, 0, 0.06);
    }}
    .section-title {{
      margin: 0 0 10px 0;
      font-size: 18px;
    }}
    .method-filter {{
      margin-bottom: 20px;
    }}
    .method-filter-actions {{
      margin-bottom: 8px;
      display: flex;
      gap: 8px;
      flex-wrap: wrap;
    }}
    .method-filter-actions button {{
      font-size: 12px;
      padding: 4px 8px;
      border: 1px solid #bbb;
      border-radius: 6px;
      background: #fff;
      cursor: pointer;
    }}
    .table-controls {{
      margin-bottom: 12px;
      border: 1px solid #ddd;
      border-radius: 8px;
      background: #f9f9f9;
      padding: 10px 12px;
    }}
    .controls-title {{
      font-size: 12px;
      font-weight: 600;
      margin-bottom: 8px;
      color: #444;
    }}
    .column-selector {{
      display: flex;
      flex-wrap: wrap;
      gap: 8px 12px;
      max-height: 140px;
      overflow-y: auto;
      padding-right: 4px;
    }}
    .column-selector label {{
      display: inline-flex;
      align-items: center;
      font-size: 12px;
      color: #333;
      gap: 4px;
      white-space: nowrap;
    }}
    #method_selector {{
      flex-direction: column;
      flex-wrap: nowrap;
      align-items: flex-start;
      gap: 6px;
      max-height: 220px;
    }}
    #method_selector label {{
      width: 100%;
    }}
    .table-wrap {{
      overflow-x: auto;
      border: 1px solid #ddd;
      border-radius: 8px;
      background: #fff;
    }}
    table {{
      width: max-content;
      min-width: 100%;
      border-collapse: collapse;
      font-size: 12px;
    }}
    th, td {{
      border: 1px solid #ddd;
      padding: 6px 8px;
      text-align: left;
      vertical-align: top;
    }}
    th {{
      background: #f2f2f2;
      position: sticky;
      top: 0;
      z-index: 1;
    }}
  </style>
</head>
<body>
  <h1>LC Method Summary</h1>
  <div class="meta">Generated at: {generated_at}</div>
  <div class="table-controls method-filter">
    <div class="controls-title">Visible Methods</div>
    <div class="method-filter-actions">
      <button type="button" id="select_all_methods">Select All</button>
      <button type="button" id="clear_all_methods">Clear All</button>
    </div>
    <div id="method_selector" class="column-selector"></div>
  </div>
  <div class="panel">
    <h2 class="section-title">LC Gradient Summary</h2>
    <div id="gradient_plot" style="width:100%;height:480px;"></div>
  </div>
  <div class="panel">
    <h2 class="section-title">LC Method Details</h2>
    <div class="table-controls">
      <div class="controls-title">Visible Columns</div>
      <div id="column_selector" class="column-selector"></div>
    </div>
    <div class="table-wrap">
      {lc_table_html}
    </div>
  </div>
  <div class="panel">
    <h2 class="section-title">MS Tune Details</h2>
    <div class="table-controls">
      <div class="controls-title">Visible Columns</div>
      <div id="ms_tune_column_selector" class="column-selector"></div>
    </div>
    <div class="table-wrap">
      {ms_tune_table_html}
    </div>
  </div>
  <div class="panel">
    <h2 class="section-title">MS Method Details</h2>
    <div class="table-controls">
      <div class="controls-title">Visible Columns</div>
      <div id="ms_method_column_selector" class="column-selector"></div>
    </div>
    <div class="table-wrap">
      {ms_method_table_html}
    </div>
  </div>
  <script>
    const traces = {traces_json};
    const defaultUncheckedColumns = new Set({unchecked_columns_json});
    const layout = {{
      title: "%B vs Time",
      xaxis: {{ title: "Time (min)" }},
      yaxis: {{ title: "%B", autorange: true }},
      template: "plotly_white",
      legend: {{ title: {{ text: "Method" }} }},
      margin: {{ l: 40, r: 40, t: 60, b: 40 }},
      shapes: [
        {{
          type: "line",
          xref: "paper",
          x0: 0,
          x1: 1,
          yref: "y",
          y0: 0,
          y1: 0,
          line: {{ color: "rgba(120, 120, 120, 0.55)", width: 1 }}
        }},
        {{
          type: "line",
          xref: "paper",
          x0: 0,
          x1: 1,
          yref: "y",
          y0: 100,
          y1: 100,
          line: {{ color: "rgba(120, 120, 120, 0.55)", width: 1 }}
        }}
      ]
    }};
    const methodSelector = document.getElementById("method_selector");

    const uniqueMethodsFromTable = () => {{
      const table = document.getElementById("lc_method_table");
      if (!table) return [];
      const names = Array.from(table.querySelectorAll("tbody tr td:first-child"))
        .map((cell) => (cell.textContent || "").trim())
        .filter((name) => name.length > 0);
      return Array.from(new Set(names));
    }};

    const selectedMethods = () => {{
      if (!methodSelector) return new Set();
      const checked = Array.from(methodSelector.querySelectorAll("input[type='checkbox']:checked"));
      return new Set(checked.map((el) => el.value));
    }};

    const filterTableRowsByMethod = (tableId, selected) => {{
      const table = document.getElementById(tableId);
      if (!table) return;
      const rows = table.querySelectorAll("tbody tr");
      rows.forEach((row) => {{
        const firstCell = row.querySelector("td:first-child");
        const methodName = (firstCell?.textContent || "").trim();
        row.style.display = selected.has(methodName) ? "" : "none";
      }});
    }};

    const applyMethodFilter = () => {{
      const selected = selectedMethods();
      const filteredTraces = traces.filter((trace) => selected.has(String(trace.name || "")));
      Plotly.react("gradient_plot", filteredTraces, layout, {{ responsive: true }});
      filterTableRowsByMethod("lc_method_table", selected);
      filterTableRowsByMethod("ms_tune_table", selected);
      filterTableRowsByMethod("ms_method_table", selected);
    }};

    const setAllMethodChecks = (checked) => {{
      if (!methodSelector) return;
      const boxes = methodSelector.querySelectorAll("input[type='checkbox']");
      boxes.forEach((box) => {{
        box.checked = checked;
      }});
      applyMethodFilter();
    }};

    const initMethodSelector = () => {{
      if (!methodSelector) return;
      const methods = uniqueMethodsFromTable();
      methods.forEach((methodName) => {{
        const label = document.createElement("label");
        const checkbox = document.createElement("input");
        checkbox.type = "checkbox";
        checkbox.value = methodName;
        checkbox.checked = true;
        checkbox.addEventListener("change", applyMethodFilter);
        label.appendChild(checkbox);
        label.appendChild(document.createTextNode(methodName));
        methodSelector.appendChild(label);
      }});

      const selectAllButton = document.getElementById("select_all_methods");
      const clearAllButton = document.getElementById("clear_all_methods");
      selectAllButton?.addEventListener("click", () => setAllMethodChecks(true));
      clearAllButton?.addEventListener("click", () => setAllMethodChecks(false));

      applyMethodFilter();
    }};

    const initColumnSelector = (tableId, selectorId) => {{
      const table = document.getElementById(tableId);
      const selector = document.getElementById(selectorId);
      if (!table || !selector) return;

      const setColumnVisibility = (colIndex, isVisible) => {{
        const nth = colIndex + 1;
        const cells = table.querySelectorAll(`tr > *:nth-child(${{nth}})`);
        cells.forEach((cell) => {{
          cell.style.display = isVisible ? "" : "none";
        }});
      }};

      const headers = Array.from(table.querySelectorAll("thead th"));
      headers.forEach((header, idx) => {{
        const name = (header.textContent || "").trim() || `column_${{idx + 1}}`;
        const label = document.createElement("label");
        const checkbox = document.createElement("input");
        checkbox.type = "checkbox";
        checkbox.checked = !defaultUncheckedColumns.has(name);
        checkbox.addEventListener("change", () => {{
          setColumnVisibility(idx, checkbox.checked);
        }});
        label.appendChild(checkbox);
        label.appendChild(document.createTextNode(name));
        selector.appendChild(label);
        setColumnVisibility(idx, checkbox.checked);
      }});
    }};

    initMethodSelector();
    initColumnSelector("lc_method_table", "column_selector");
    initColumnSelector("ms_tune_table", "ms_tune_column_selector");
    initColumnSelector("ms_method_table", "ms_method_column_selector");
  </script>
</body>
</html>
"""
    output_path.write_text(html, encoding="utf-8")


def main() -> None:
    root_dir = Path.cwd()
    method_dirs = discover_method_dirs(root_dir)
    if not method_dirs:
        warn(f"No method directories matching *.m found in {root_dir}")
        raise SystemExit(1)

    summaries: list[MethodSummary] = []
    for method_dir in method_dirs:
        summary = parse_method_dir(method_dir)
        if summary is not None:
            summaries.append(summary)

    if not summaries:
        warn("No readable methods were found after parsing.")
        raise SystemExit(1)

    df = pd.DataFrame(asdict(summary) for summary in summaries)
    df.sort_values("method_dir", inplace=True)
    to_csv(df, root_dir / OUTPUT_CSV)

    traces = build_gradient_traces(df)
    to_html(df, traces, root_dir / OUTPUT_HTML)

    print(f"Wrote {len(df)} rows to {OUTPUT_CSV} and {OUTPUT_HTML}")


# =========================
# Run on uploaded ZIP files
# =========================
OUTPUT_DIR = Path('/content/sample_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Upload ZIP file(s) that contain *.m method folders...')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No files uploaded.')

zip_items = [(name, data) for name, data in uploaded.items() if name.lower().endswith('.zip')]
if not zip_items:
    raise RuntimeError('Please upload at least one .zip file.')

summaries: list[MethodSummary] = []
with tempfile.TemporaryDirectory(prefix='lcms_zip_', dir='/content') as tmp_dir:
    tmp_root = Path(tmp_dir)
    extract_root = tmp_root / 'extracted'
    extract_root.mkdir(parents=True, exist_ok=True)

    for zip_name, zip_bytes in zip_items:
        zip_path = tmp_root / zip_name
        zip_path.write_bytes(zip_bytes)
        dst = extract_root / Path(zip_name).stem
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as zf:
            zf.extractall(dst)

    method_dirs = sorted(p for p in extract_root.rglob('*.m') if p.is_dir())
    print(f'Found *.m dirs recursively: {len(method_dirs)}')
    for p in method_dirs[:30]:
        print(' -', p.relative_to(extract_root))

    if not method_dirs:
        raise RuntimeError('No *.m directories found in uploaded ZIP(s).')

    for method_dir in method_dirs:
        summary = parse_method_dir(method_dir)
        if summary is None:
            continue
        summary.method_dir = str(method_dir.relative_to(extract_root))
        summaries.append(summary)

# Temporary extracted data has been removed automatically here.
if not summaries:
    raise RuntimeError('No readable methods were found after parsing.')

df = pd.DataFrame(asdict(s) for s in summaries)
df.sort_values('method_dir', inplace=True)

CSV_PATH = OUTPUT_DIR / OUTPUT_CSV
HTML_PATH = OUTPUT_DIR / OUTPUT_HTML

to_csv(df, CSV_PATH)
traces = build_gradient_traces(df)
to_html(df, traces, HTML_PATH)

print(f'Wrote {len(df)} rows')
print(f'CSV: {CSV_PATH}')
print(f'HTML: {HTML_PATH}')

from IPython.display import HTML as _HTML, display  # noqa: E402

display(_HTML(f'<p><b>CSV:</b> {CSV_PATH}</p>'))
if HTML_PATH.exists():
    display(_HTML(HTML_PATH.read_text(encoding='utf-8')))

files.download(str(CSV_PATH))
files.download(str(HTML_PATH))

